<a href="https://colab.research.google.com/github/aymanr8-dot/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026_09_25-cleaning_gauntlet-lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [30]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [31]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [32]:
before = len(df)
df = df.drop_duplicates()
log('drop_duplicates', 'removed exact duplicate rows', before - len(df))
print('rows after dedup:', len(df))

[drop_duplicates] removed exact duplicate rows (15 row(s))
rows after dedup: 300


### TODO 2 — clean `price` -> float

In [33]:
df['price'] = df['price'].str.replace('$', '', regex=False).astype(float)
log('clean_price', 'stripped $ and cast price to float', len(df))

[clean_price] stripped $ and cast price to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [34]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
missing = df['qty'].isna().sum()
negative = (df['qty'] < 0).sum()

df_before_drop = df.copy()   # snapshot for the revenue before/after comparison

before_drop = len(df)
df = df[df['qty'] >= 1].copy()
dropped = before_drop - len(df)

log('qty missing', f'dropped {missing} row(s) with missing quantity', missing)
log('qty negative', f'dropped {negative} row(s) with negative quantity (refunds)', negative)

# revenue before vs after this step, for the write-up
before_revenue = (df_before_drop['qty'] * df_before_drop['price']).sum()
after_revenue = (df['qty'] * df['price']).sum()
print('revenue before drop:', round(before_revenue, 2))
print('revenue after drop:', round(after_revenue, 2))

[qty missing] dropped 12 row(s) with missing quantity (12 row(s))
[qty negative] dropped 13 row(s) with negative quantity (refunds) (13 row(s))
revenue before drop: 4594.5
revenue after drop: 4740.0


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [35]:
print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho',
}

before_items = df['item'].nunique()
df['item'] = df['item'].map(ITEM_MAP)
after_items = df['item'].nunique()

log('item', f'collapsed {before_items} spelling variants into {after_items} canonical products', len(df))
print(df['item'].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] collapsed 6 spelling variants into 3 canonical products (275 row(s))
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [36]:
print(df['category'].value_counts())

df['category'] = df['category'].str.lower().str.strip().str.replace('-', '', regex=False)

CATEGORY_MAP = {
    'food': 'Food',
    'merch': 'Merch',
    'apparel': 'Merch',   # business decision: folding Apparel into Merch —
                            # not a spelling issue, a judgment call that
                            # a T-shirt/apparel item counts as merchandise
    'raingear': 'RainGear',
}
before_cats = df['category'].nunique()
df['category'] = df['category'].map(CATEGORY_MAP)
after_cats = df['category'].nunique()

log('category', f'normalized casing/punctuation, merged Apparel into Merch as a business decision — {before_cats} variants collapsed to {after_cats} categories', len(df))
print(df['category'].value_counts())

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] normalized casing/punctuation, merged Apparel into Merch as a business decision — 4 variants collapsed to 3 categories (275 row(s))
category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [37]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert set(df['item'].unique()) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
assert set(df['category'].unique()) == {'Food', 'Merch', 'RainGear'}
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [38]:
df['revenue'] = (df['price'] * df['qty']).round(2)
rev_by_cat = df.groupby('category')['revenue'].sum().round(2).sort_values(ascending=False)
print(rev_by_cat)
print('TOTAL:', round(df['revenue'].sum(), 2))

category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
TOTAL: 4740.0


**What I would tell the vendor:** Stock more Food (e.g. Cheeseburgers) as it is the top revenue category at 1656.00, ahead of Merch (1572.00) and RainGear (1512.00).

### TODO 8 — read back your log

In [39]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,drop_duplicates,removed exact duplicate rows,15
1,clean_price,stripped $ and cast price to float,300
2,qty missing,dropped 12 row(s) with missing quantity,12
3,qty negative,dropped 13 row(s) with negative quantity (refu...,13
4,item,collapsed 6 spelling variants into 3 canonical...,275
5,category,"normalized casing/punctuation, merged Apparel ...",275


In [40]:
rng2 = np.random.default_rng(5)
rows2 = [{'order_id': i, 'item': rng2.choice(items), 'category': rng2.choice(cats), 'qty': rng2.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]), 'price': rng2.choice(['$7.50','7.5','$12.00','24','6.0'])} for i in range(300)]
r = pd.DataFrame(rows2)
r = pd.concat([r, r.sample(15, random_state=1)])
def rev(x): return round((pd.to_numeric(x['price'].astype(str).str.replace('$','',regex=False)) * pd.to_numeric(x['qty'], errors='coerce')).sum(), 2)
print('raw:', rev(r))
r2 = r.drop_duplicates()
print('after drop_duplicates:', rev(r2))
r3 = r2.copy(); r3['price'] = r3['price'].astype(str).str.replace('$','',regex=False).astype(float)
print('after clean_price:', round((r3['price']*pd.to_numeric(r3['qty'],errors='coerce')).sum(),2))
r4 = r3[pd.to_numeric(r3['qty'],errors='coerce').notna() & (pd.to_numeric(r3['qty'],errors='coerce')>=1)].copy()
print('after clean_qty:', round((r4['price']*r4['qty'].astype(float)).sum(),2))

raw: 4852.5
after drop_duplicates: 4594.5
after clean_price: 4594.5
after clean_qty: 4740.0


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

**a)

Revenue total before drop_duplicates (raw frame): 4852.50.

Revenue total after drop_duplicates: 4594.50.

The revenue changed by -258.000. The 15 injected duplicate rows were double-counting real orders, so removing them shifted the reported number most.

**b)

I merged the 'Apparel' category into 'Merch', treating items such as the Foam Finger as merchandise. A reasonable person could have kept 'Apparel' as its own category. Since revenue is summed per category, splitting merch and apparel apart would not change the overall total (remaining at 4740.00) however, it would divide the current Merch figure of 1572.00 across two smaller categoriesm which could change the category suggested to be the top seller. This could in turn affect the recommendation going out to the selller.

I merged the 'Apparel' and 'Merch' as the three real products map cleanly to Food / Merch / Raingear, a single Foam Finger product in between apparel and merch labels is a labeling artifact rather than a business distinction. As a result, these three categories help keep the report legible and defensible.